##Step 1: Install dependencies

In [2]:
!pip install -q sentence-transformers datasets pinecone weaviate-client qdrant-client chromadb

import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import pinecone
import weaviate
from qdrant_client import QdrantClient
from qdrant_client.http import models
import chromadb
import uuid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.3 MB/s eta 0:00:00


## Step 2: Load and prepare the Quora Question Pairs dataset
Using a subset of 1000 questions for demo, with topic metadata

In [4]:
dataset = load_dataset("squad", split="train[:1000]")
questions = [item['question'] for item in dataset]
# Create synthetic metadata for filtering (e.g., assign categories based on context length)
metadata = [{"id": str(uuid.uuid4()), "category": "long" if len(item['context'].split()) > 100 else "short"}
            for item in dataset]

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

## Step 3: Initialize the embedding model
Using all-MiniLM-L6-v2 for 384-dimensional embeddings

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(questions, batch_size=32, show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

## Step 4: ChromaDB - Lightweight Embeddable Vector Database


In [8]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="squad_questions")

In [9]:
# Add vectors with metadata
collection.add(
    embeddings=embeddings.tolist(),
    documents=questions,
    metadatas=metadata,
    ids=[meta["id"] for meta in metadata]
)

In [15]:
query = "Pakistan"
query_emb = model.encode([query])[0]

In [16]:
# Query ChromaDB
chroma_results = collection.query(
    query_embeddings=query_emb.tolist(),
    n_results=3,
    where={"category": "short"}
)
print("\nChromaDB Results:")
for id, doc, dist in zip(chroma_results["ids"][0], chroma_results["documents"][0], chroma_results["distances"][0]):
    print(f"ID: {id}, Text: {doc}, Distance: {dist}")



ChromaDB Results:
ID: 2455e41e-0767-430a-af25-e34ce080321a, Text: Her first appearance performing since giving birth was where?, Distance: 1.529323697090149
ID: 3b108aeb-0781-44a9-b924-a454829e51f8, Text: What did Beyonce and Rowland found in 2005?, Distance: 1.5603559017181396
ID: e4915b57-b084-4265-81d3-d28a5d7929a6, Text: Who released the information about Beyoncé's performance for the Libyan ruler?, Distance: 1.565186858177185
